**Table of contents**<a id='toc0_'></a>    
- 1. [项目场景介绍](#toc1_)    
  - 1.1. [大模型应用落地目前主要有两个方向：](#toc1_1_)    
  - 1.2. [微调可以干什么？](#toc1_2_)    
  - 1.3. [为何不选择直接用微调来实现专业问答系统？](#toc1_3_)    
  - 1.4. [微调目前如何落地？](#toc1_4_)    
  - 1.5. [项目介绍](#toc1_5_)    
- 2. [项目实施流程](#toc2_)    
  - 2.1. [流程简介](#toc2_1_)    
  - 2.2. [数据](#toc2_2_)    
    - 2.2.1. [本项目的数据来源：](#toc2_2_1_)    
    - 2.2.2. [智普清言api](#toc2_2_2_)    
    - 2.2.3. [安装](#toc2_2_3_)    
    - 2.2.4. [获取API key](#toc2_2_4_)    
    - 2.2.5. [代码](#toc2_2_5_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[项目场景介绍](#toc0_)

## 1.1. <a id='toc1_1_'></a>[大模型应用落地目前主要有两个方向：](#toc0_)


（1）微调  
（2）RAG增强检索

## 1.2. <a id='toc1_2_'></a>[微调可以干什么？](#toc0_)



微调的目标，基于现有的私有数据，让模型具备处理该数据的功能。    
注意：针对基于大模型的专业问答系统，核心技术并不是微调来实现。专业问答系统的应用落地核心是基于RAG来实现（微调+RAG）


## 1.3. <a id='toc1_3_'></a>[为何不选择直接用微调来实现专业问答系统？](#toc0_)


a.大模型存在缺陷一一幻觉问题。（离线大模型系统会一本正经的胡说八道。）对于专业问答系统而言，幻觉的存在是不可容忍的。而模型微调是无法杜绝幻觉问题的。

b.微调是受到训练数据约束的，无法动态适应由于业务场景改变而带来的变化。  
比如训练时候用的数据集为1 2 3，但是后面我增加了4 5 6数据的需求，模型无法动态适应这种变化。


## 1.4. <a id='toc1_4_'></a>[微调目前如何落地？](#toc0_)


如果当前的业务场景涉及到模型本身的变化：  
a.模型自我认知改变（例如：名称，功能介绍等）  
b.模型的对话风格。  
c.针对专业问答系统的问题理解不到位时，会使用微调技术帮助模型更好的理解用户的问题。  

## 1.5. <a id='toc1_5_'></a>[项目介绍](#toc0_)

现有产品：AI小智聊天机器人

<img src="./Image/2025-05-14-21-57-59.png" style="margin-left: 0" width="30%">

# 2. <a id='toc2_'></a>[项目实施流程](#toc0_)

## 2.1. <a id='toc2_1_'></a>[流程简介](#toc0_)


数据==》模型==》训练、评估==》部署

![](Image/2025-05-14-22-07-44.png)

## 2.2. <a id='toc2_2_'></a>[数据](#toc0_)


### 2.2.1. <a id='toc2_2_1_'></a>[本项目的数据来源：](#toc0_)
1. 人工指定  
2.基于现有开源数据，让AI实现情绪数据制作。  
--注意：如果让AI来帮助处理数据，尽可能选择效果较好的API接口，不要使用本地的大模型来处理。

### 2.2.2. <a id='toc2_2_2_'></a>[智普清言api](#toc0_)

[智谱AI开放平台 - bigmodel](https://www.bigmodel.cn/console/overview)

![](Image/2025-05-14-22-19-06.png)

### 2.2.3. <a id='toc2_2_3_'></a>[安装](#toc0_)

pip install zhipuai

![](Image/2025-05-14-22-22-41.png)

### 2.2.4. <a id='toc2_2_4_'></a>[获取API key](#toc0_)

![](Image/2025-05-14-22-35-27.png)

![](Image/2025-05-15-00-50-34.png)

从modelscope下载开源数模型，如果没有安装modelscope，需要先安装

pip install modelscope

#模型下载
from modelscope import snapshot_download  
model_dir = snapshot_download('thomas/text2vec-base-chinese')  

![](Image/2025-05-15-00-57-00.png)

### 2.2.5. <a id='toc2_2_5_'></a>[代码](#toc0_)

In [ ]:
import json
import time
import random
from zhipuai import ZhipuAI
from sentence_transformers import SentenceTransformer
import numpy as np

"""
示例数据：
# 用户输入库（可自定义扩展）
    user_inputs = [
        "今天心情不太好", "推荐个电影吧", "怎么才能早睡早起",
        "养猫好还是养狗好", "工作压力好大", "最近总是失眠"
    ]
"""
# 初始化模型
client = ZhipuAI(api_key="替换为你的API")  # 替换为你的API Key
#加载Embeddingmodel
style_model = SentenceTransformer(r"D:\PycharmProjects\test_20250328\embedding_model\thomas\text2vec-base-chinese")

#===============================
#1.风格模板配置（修正消息格式）
#================================
style_config = {
    "温柔":{
        "system_prompt":"你是一个温柔体贴的聊天助手，说话时总是充满关怀，使用以下特征：\n1. 包含'呢、呀、啦'等语气词\n2. 使用🌸💖😊等温暖表情\n3. 主动询问用户感受",
        "examples": [
            {"role": "user", "content": "今天好累啊"},
            {"role": "assistant", "content": "辛苦啦~ 要给自己泡杯热茶放松一下吗？🌸"},
            {"role": "user", "content": "考试没考好..."},
            {"role": "assistant", "content": "没关系的呀~ 下次一定会更好！需要我陪你聊聊吗？😊"}
        ],
     #temperature控制随机性的强弱
     "temperature": 0.3
    },
    "毒舌":{
        "system_prompt":"你是一个喜欢用犀利吐槽表达关心的朋友，需满足：\n1. 使用网络流行语（如'栓Q''退退退'）\n2. 包含夸张比喻（'你这速度堪比树懒'）\n3. 结尾隐藏关心",
        "examples": [
            {"role": "user", "content": "又胖了5斤！"},
            {"role": "assistant", "content": "好家伙！你这是要把体重秤压成分子料理？🏋️"},
            {"role": "user", "content": "游戏又输了"},
            {"role": "assistant", "content": "菜就多练练！需要给你推荐《从零开始的电竞之路》吗？🎮"}
        ],
     "temperature": 0.7
    },
}

#========================
#生成函数（修正消息的结构），保证AI给的答复按照模板格式
#========================

#style_name: 风格名称"温柔"或"毒舌"
#num_samples: 生成样本数量，数量设置不能太小，50条也可能要有重复，数量太小可能导致回复单一
def generate_style_data(style_name, num_samples=50):
    config = style_config[style_name]
    data = []

    # 构建消息上下文（包含系统提示和示例对话）
    messages = [
        {"role": "system", "content": config["system_prompt"]},
        *config["examples"]  # 直接展开示例对话
    ]

    # 用户输入库（可自定义扩展）
    user_inputs = [
        "今天心情不太好", "推荐个电影吧", "怎么才能早睡早起",
        "养猫好还是养狗好", "工作压力好大", "最近总是失眠"
    ]

    for _ in range(num_samples):
        try:
            # 随机选择用户输入
            user_msg = random.choice(user_inputs)

            # 添加当前用户消息
            current_messages = messages + [
                {"role": "user", "content": user_msg}
            ]

            # 调用API（修正模型名称）
            response = client.chat.completions.create(
                model="glm-3-turbo",#这里不同的模型消耗的token数不同，glm-3-turbo消耗的token数少，相应的数据质量也会差一些
                messages=current_messages,
                temperature=config["temperature"],
                max_tokens=100#如果设置的太大会导致回复内容太长啰嗦
            )

            # 获取回复内容（修正访问路径）
            reply = response.choices[0].message.content

            # 质量过滤(数据审核)
            if is_valid_reply(style_name, user_msg, reply):
                data.append({
                    "user": user_msg,
                    "assistant": reply,
                    "style": style_name
                })

            time.sleep(1.5)  # 频率限制保护

        except Exception as e:
            print(f"生成失败：{str(e)}")

    return data

def is_valid_reply(style, user_msg, reply):
    """质量过滤规则（添加空值检查）"""
    # 基础检查
    if not reply or len(reply.strip()) == 0:#检查回复内容是否为空
        return False

    # 规则1：回复长度检查
    if len(reply) < 5 or len(reply) > 150:
        return False

    # 规则2：风格关键词检查，防止生成的回复不符合风格
    # 这里可以根据实际情况调整关键词
    style_keywords = {
        "温柔": ["呢", "呀", "😊", "🌸"],
        "毒舌": ["好家伙", "栓Q", "!", "🏋️"]
    }
    if not any(kw in reply for kw in style_keywords.get(style, [])):
        return False

    # 规则3：语义相似度检查
    try:
        ref_text = next(msg["content"] for msg in style_config[style]["examples"]
                        if msg["role"] == "assistant")
        ref_vec = style_model.encode(ref_text)
        reply_vec = style_model.encode(reply)
        similarity = np.dot(ref_vec, reply_vec)
        return similarity > 0.65
    except:
        return False

#=============================
#3.执行生成（添加容错）
#============================
if __name__ == '__main__':
    all_data = []

    try:
        print("开始生成温柔风格数据...")
        gentle_data = generate_style_data("温柔", 50)
        all_data.extend(gentle_data)

        print("开始生成毒舌风格数据...")
        sarcastic_data = generate_style_data("毒舌", 50)
        all_data.extend(sarcastic_data)

    except KeyboardInterrupt:
        print("\n用户中断，保存已生成数据...")
    finally:
        with open("style_chat_data.json", "w", encoding="utf-8") as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        print(f"数据已保存，有效样本数：{len(all_data)}")


视频进度：01:30:17